# Part C: fixed ensembles, then two more seeds per target

Run one experiment cell at a time on a **free Colab T4**. Each training cell has a 110-minute timeout; rerun that cell after a disconnect to resume. Do not use “Run all” for the four training cells in one free session.

Long phases show a small progress bar with percentage, elapsed time and ETA. ETA covers the displayed phase or epoch, not the entire experiment. It settles after a few batches; resumed runs start at their saved position.

This notebook clones **fix/runnable-pipeline**, and reuses `MyDrive/hallu_poc/work`. Push the reviewed local commits before running it. It does not modify the original Part A/B predictions. Every experiment prints a table to paste back.

Members, equal weights, seeds (43 and 44), three epochs, 4,096 context and decoder grid are fixed in advance. No member/weight search on test results. Our models tune decoding on the existing source-held-out dev split. Mixtures containing published LettuceDetect use fixed decoding because its training included our dev sources. Cross-fitting remains restricted to published LettuceDetect alone.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, subprocess
from pathlib import Path
REPO = Path('/content/tse')
BRANCH = 'fix/runnable-pipeline'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH,
                    'https://github.com/gbirhanu/tse.git', str(REPO)], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip() == BRANCH
REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
PROJECT = REPO / 'hallu_poc'
WORK = Path('/content/drive/MyDrive/hallu_poc/work')
DATA = Path('/content/drive/MyDrive/hallu_poc/RAGTruth')
if not (DATA / 'response.jsonl').exists() and not (DATA / 'dataset/response.jsonl').exists():
    DATA = Path('/content/RAGTruth')
    if not DATA.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
print('Code revision:', REVISION, '\\nOutputs:', WORK)


### Optional: update code in an existing Colab session

After pushing new commits from your computer, let any running experiment finish before running the next cell. It updates the code for subsequent experiments and preserves saved outputs on Drive. A run already in progress continues using the old code. Skip this cell if you just cloned the latest version.


In [ ]:
assert subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip() == BRANCH
subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Updated code revision:', REVISION)


In [ ]:
def run(*arguments, timeout=110 * 60):
    command = [sys.executable, '-u', str(PROJECT / 'poc.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    try:
        subprocess.run(command, cwd=PROJECT, check=True, timeout=timeout,
                       env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false'})
    except subprocess.TimeoutExpired:
        raise RuntimeError('110-minute budget reached. Checkpoints are on Drive; reconnect and rerun this cell.')

def accepted_reference(run_name, fallback):
    path = WORK / 'results' / f'{run_name}_levers.json'
    if not path.exists():
        return fallback
    result = json.loads(path.read_text())
    accepted = [r for r in result['rows'] if r.get('decision') == 'keep']
    return f"{run_name}:{accepted[0]['setting']}" if accepted else fallback

def stage1_reference():
    best = accepted_reference('ens_hard_soft_s42', 'ld_large:baseline')
    return accepted_reference('ens_ld_hard_soft_s42', best)

LEGACY = ['--legacy-tokenizer', 'answerdotai/ModernBERT-base']
help_text = subprocess.check_output([sys.executable, str(PROJECT / 'poc.py'), '--help'], text=True)
assert '--members' in help_text, 'Push the new commits, then restart with a fresh /content checkout.'
run('report')


## 1a. Average the existing hard and soft models; tune on dev

Compare once against the original published LD baseline. A strictly positive paired 95% interval is required for **keep**. A rejected experiment remains recorded for transparency and does not become the reference.


In [ ]:
run('ensemble', '--name', 'ens_hard_soft_s42', '--members', 'mb_hard', 'mb_soft',
    '--fit', 'dev', '--against', 'ld_large:baseline', *LEGACY)


## 1b. Add published LettuceDetect using fixed decoding

Equal weights and the original 0.5 threshold are declared before evaluation. There is no clean RAGTruth dev set for this mixture, so this cell cannot tune its decoder. This is a separate fixed-decoding candidate, not a held-out-calibrated claim.


In [ ]:
reference = accepted_reference('ens_hard_soft_s42', 'ld_large:baseline')
run('ensemble', '--name', 'ens_ld_hard_soft_s42', '--members', 'ld_large', 'mb_hard', 'mb_soft',
    '--fit', 'fixed', '--against', reference, *LEGACY)


## 2. Four independent training cells (one per session if needed)

Each uses the same settings as Part B and always saves checkpoints, the best dev model and prediction progress to Drive. A completed cell is validated and skipped on rerun. A changed configuration requires a new run name.

**Soft-target compatibility:** these two new soft runs deliberately preserve Part B's legacy edge-shift implementation. An audit found inward shifts can exceed five words for long spans. Correcting that is a separate future ablation; changing it here would confound the seed comparison. Source-bootstrap intervals do not quantify all training-seed uncertainty.


### Hard targets, seed 43

Run this cell alone. The table is descriptive; individual seeds do not change the reference or choose ensemble members.


In [ ]:
run('train', '--name', 'mb_hard_s43', '--base', 'answerdotai/ModernBERT-base',
    '--targets', 'hard', '--seed', '43', '--epochs', '3', '--lr', '2e-5',
    '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200', '--device', 'cuda')
run('levers', '--name', 'mb_hard_s43', '--fit', 'dev')


### Soft targets, seed 43

Run this cell alone. The table is descriptive; individual seeds do not change the reference or choose ensemble members.


In [ ]:
run('train', '--name', 'mb_soft_s43', '--base', 'answerdotai/ModernBERT-base',
    '--targets', 'soft', '--seed', '43', '--epochs', '3', '--lr', '2e-5',
    '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200', '--device', 'cuda')
run('levers', '--name', 'mb_soft_s43', '--fit', 'dev')
run('report', '--compare', 'mb_hard_s43', 'mb_soft_s43')


### Hard targets, seed 44

Run this cell alone. The table is descriptive; individual seeds do not change the reference or choose ensemble members.


In [ ]:
run('train', '--name', 'mb_hard_s44', '--base', 'answerdotai/ModernBERT-base',
    '--targets', 'hard', '--seed', '44', '--epochs', '3', '--lr', '2e-5',
    '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200', '--device', 'cuda')
run('levers', '--name', 'mb_hard_s44', '--fit', 'dev')


### Soft targets, seed 44

Run this cell alone. The table is descriptive; individual seeds do not change the reference or choose ensemble members.


In [ ]:
run('train', '--name', 'mb_soft_s44', '--base', 'answerdotai/ModernBERT-base',
    '--targets', 'soft', '--seed', '44', '--epochs', '3', '--lr', '2e-5',
    '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200', '--device', 'cuda')
run('levers', '--name', 'mb_soft_s44', '--fit', 'dev')
run('report', '--compare', 'mb_hard_s44', 'mb_soft_s44')


## 3. Frozen three-seed ensembles and the soft-target comparison

All runs must be complete. Evaluate hard seeds 42/43/44, then soft seeds 42/43/44, against the accepted reference at each step. The separate soft-minus-hard comparison answers whether soft targets help. Paired intervals resample test sources; report the three matched-seed comparisons as well.


In [ ]:
run('ensemble', '--name', 'ens_hard_3seed', '--members', 'mb_hard', 'mb_hard_s43', 'mb_hard_s44',
    '--fit', 'dev', '--against', stage1_reference(), *LEGACY)
run('ensemble', '--name', 'ens_soft_3seed', '--members', 'mb_soft', 'mb_soft_s43', 'mb_soft_s44',
    '--fit', 'dev', '--against', accepted_reference('ens_hard_3seed', stage1_reference()), *LEGACY)
run('report', '--compare', 'ens_hard_3seed', 'ens_soft_3seed')


## 4. Add the soft seeds to the hard-seed ensemble

One predeclared equal-weight ensemble of all six own models; tune only on dev. This is the last candidate in Part C. Keep the accepted reference if this interval includes zero. Paste the final table and matched-seed differences back before adding any new loss, context length or pretraining data.


In [ ]:
reference = accepted_reference('ens_soft_3seed', accepted_reference('ens_hard_3seed', stage1_reference()))
run('ensemble', '--name', 'ens_hard_soft_6seed', '--members',
    'mb_hard', 'mb_hard_s43', 'mb_hard_s44', 'mb_soft', 'mb_soft_s43', 'mb_soft_s44',
    '--fit', 'dev', '--against', reference, *LEGACY)
run('report')
for hard, soft in [('mb_hard', 'mb_soft'), ('mb_hard_s43', 'mb_soft_s43'), ('mb_hard_s44', 'mb_soft_s44')]:
    run('report', '--compare', hard, soft)


## Reading the result and next experiments

The main metric remains LettuceDetect-compatible character span F1. **Keep** requires a positive candidate-minus-reference gain and a strictly positive lower 95% paired source-bootstrap bound; otherwise retain the reference. These are predeclared exploratory comparisons, not multiplicity-adjusted proof.

The extra set/task-macro column is important for later paper comparisons: RLSeek's 60.1 and RL4HS's 58.3 are task averages, whereas our main LD score pools character counts across tasks. Crossing 60.1 in our LD column alone is not yet an apples-to-apples claim of beating RLSeek. [RLSeek paper](https://aclanthology.org/2026.acl-long.1492.pdf), [RL4HS paper](https://arxiv.org/html/2510.02173v2).

After reviewing this table, test one change at a time: character-weighted differentiable F1 loss; then 8,192 context with a measured T4 memory preflight; then English PsiloQA pretraining; then a summary-specific intervention chosen from dev errors. No later experiment is enabled in this notebook.
